# NYC Tabblock Analysis

**Goal:** Use Homecastr's tabblock-level forecasts to analyze sub-neighborhood price dynamics across New York City.

Homecastr has forecasts for **32,892 NYC tabblocks** — roughly city-block granularity — using the NYC PLUTO jurisdiction model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/neighborhoods/01_nyc_tabblock_analysis.ipynb)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from homecastr import HomecastrClient

client = HomecastrClient(os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly"))

## Understanding GEOID20s for NYC

NYC tabblock GEOIDs follow: `36` (NY state) + `005` (Richmond/Staten Island county) + 6-digit tract + 4-digit block

Borough FIPS codes:
| Borough | County FIPS |
|---|---|
| Manhattan | 061 |
| Bronx | 005 |
| Brooklyn | 047 |
| Queens | 081 |
| Staten Island | 085 |

In [ ]:
# Confirmed live tabblock GEOIDs from Homecastr database
# All are in Staten Island (county 005). 
# To get GEOIDs for other boroughs, use the Census TIGER/Line API:
#   https://geocoding.geo.census.gov/geocoder/

NYC_TABBLOCKS = [
    "360050001001001",  # Tract 1, Block 1001
    "360050002000001",  # Tract 2, Block 0001
    "360050002001000",  # Tract 2, Block 1000
    "360050002001001",  # Tract 2, Block 1001
    "360050002001002",  # Tract 2, Block 1002
]

print("Fetching NYC tabblock forecasts...")
df = client.forecast.by_tabblock.retrieve_many(NYC_TABBLOCKS)
print(df)

In [ ]:
# Pull full fan chart for one tabblock
result = client.forecast.by_tabblock.retrieve("360050002000001")
fan_df = pd.DataFrame(result["fan_chart"])

print(f"Tabblock: {result['geoid']}")
print(f"Jurisdiction: {result['jurisdiction']}")
print(f"Current value: ${result['current_value']:,}")
print(f"1-year growth: {result.get('growth_pct', 'n/a')}%")
print()
print(fan_df[["year", "horizon_months", "p10", "p50", "p90", "growth_pct"]].to_string(index=False))

In [ ]:
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(fan_df["year"], fan_df["p10"] / 1e3, fan_df["p90"] / 1e3,
                alpha=0.15, color="crimson", label="P10–P90")
ax.fill_between(fan_df["year"], fan_df["p25"] / 1e3, fan_df["p75"] / 1e3,
                alpha=0.3, color="crimson", label="P25–P75")
ax.plot(fan_df["year"], fan_df["p50"] / 1e3, color="crimson", lw=2, label="P50 (median)")
ax.axhline(result["current_value"] / 1e3, color="gray", ls="--", lw=1, label="Current value")

ax.set_title(f"NYC Tabblock Forecast — GEOID {result['geoid']}")
ax.set_xlabel("Year")
ax.set_ylabel("Value ($000s)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}k"))
ax.legend()
plt.tight_layout()
plt.show()

## Cross-Borough ZCTA Comparison

For a broader NYC view across all 5 boroughs, use ZCTA-level forecasts.

In [ ]:
NYC_ZIPS = {
    "Manhattan - Midtown (10001)": "10001",
    "Manhattan - UWS (10024)": "10024",
    "Brooklyn - Williamsburg (11211)": "11211",
    "Brooklyn - Park Slope (11215)": "11215",
    "Queens - Astoria (11102)": "11102",
    "Queens - Flushing (11354)": "11354",
    "Bronx - Riverdale (10463)": "10463",
    "Staten Island - St. George (10301)": "10301",
}

df_z = client.forecast.by_zcta.retrieve_many(list(NYC_ZIPS.values()))
df_z.insert(0, "neighborhood", list(NYC_ZIPS.keys()))

fig, ax = plt.subplots(figsize=(11, 5))
df_z_s = df_z.sort_values("p50_60m", ascending=True)
ax.barh(df_z_s["neighborhood"], df_z_s["p50_60m"] / 1e3, color="crimson", alpha=0.8)
ax.set_xlabel("5-Year Median Forecast ($000s)")
ax.set_title("NYC Cross-Borough 5-Year Outlook (ZCTA)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}k"))
plt.tight_layout()
plt.show()

print(df_z[["neighborhood", "current_value", "p50_12m", "p50_60m"]].to_string(index=False))